# EXP-32: SPINK1 N34S Negative Control via FEP
**Pancreatitis-associated mutation at non-interface position**

- **Expected:** |ΔΔG| < 1.0 kcal/mol (neutral for binding)
- **PASS:** |ΔΔG| < 1.0 | **MARGINAL:** |ΔΔG| < 2.0 | **FAIL:** |ΔΔG| ≥ 2.0
- **Loads SPINK1–Trypsin system from EXP-06 (or builds from 1TGS)**

In [ ]:
# Install OpenMM + all scientific dependencies (pip — OpenCL platform)
!pip install openmm mdtraj 'pymbar==4.0.3' mdanalysis
!pip install netCDF4 git+https://github.com/choderalab/openmmtools.git

# mpiplus stub — not on PyPI; minimal shim for imports
import os as _os, sys as _sys
_sp = _os.path.join(_sys.prefix, 'lib',
    f'python{_sys.version_info.major}.{_sys.version_info.minor}',
    'site-packages', 'mpiplus')
_os.makedirs(_sp, exist_ok=True)
with open(_os.path.join(_sp, '__init__.py'), 'w') as _f:
    _f.write(
        'class delayed_termination:\n'
        '    def __init__(self, func=None):\n'
        '        self._func = func\n'
        '    def __enter__(self): return self\n'
        '    def __exit__(self, *a): pass\n'
        '    def __call__(self, *args, **kwargs):\n'
        '        if self._func is not None: return self._func(*args, **kwargs)\n'
        '        if len(args) == 1 and callable(args[0]): return args[0]\n'
        '        return self\n'
        'def get_mpicomm(): return None\n'
        'def run_single_node(rank=0, broadcast_result=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def on_single_node(rank=0, broadcast_result=False, sync_nodes=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def distribute(func, seq, *args, send_results_to=None, sync_nodes=False, group_nodes=None):\n'
        '    results = [func(item, *args) for item in seq]\n'
        '    return list(zip(*results)) if results and isinstance(results[0], tuple) else (results, list(seq))\n'
    )
for _k in list(_sys.modules):
    if 'mpiplus' in _k:
        del _sys.modules[_k]

!pip install -q scipy matplotlib pandas requests gemmi

# Verify critical imports
import importlib
all_ok = True
for pkg in ['openmm', 'openmmtools', 'mdtraj', 'pymbar', 'MDAnalysis']:
    try:
        m = importlib.import_module(pkg)
        print(f"\u2713 {pkg} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"\u2717 {pkg}: {e}")
        all_ok = False

if all_ok:
    print("\n\u2705 All packages installed successfully!")
else:
    raise RuntimeError("Package installation failed \u2014 check error messages above")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)
import sys, os, shutil, json, zipfile, logging
from pathlib import Path
import numpy as np

PIPELINE_ROOT = Path('/content/drive/MyDrive/spink7_pipeline')
assert PIPELINE_ROOT.exists()
sys.path.insert(0, str(PIPELINE_ROOT))

EXP_ID = 'EXP-32'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
for d in ['prep','fep_complex','fep_free','analysis','figures']:
    (WORK_DIR/d).mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO)

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

In [ ]:
# Checkpoint & auto-save utilities
import json, shutil, threading, time as _time
from pathlib import Path

class ExperimentCheckpoint:
    """Drive-backed phase checkpoint for resilient experiment execution."""

    def __init__(self, drive_output: Path, exp_id: str):
        self.drive_output = drive_output
        self.exp_id = exp_id
        self.path = drive_output / 'experiment_checkpoint.json'
        self.state = self._load()

    def _load(self) -> dict:
        if self.path.exists():
            with open(self.path) as f:
                return json.load(f)
        return {'exp_id': self.exp_id, 'phases': {}}

    def save(self):
        self.drive_output.mkdir(parents=True, exist_ok=True)
        with open(self.path, 'w') as f:
            json.dump(self.state, f, indent=2, default=str)

    def is_done(self, phase: str) -> bool:
        return phase in self.state['phases']

    def mark_done(self, phase: str, data: dict = None):
        self.state['phases'][phase] = {
            'timestamp': _time.strftime('%Y-%m-%d %H:%M:%S'),
            'data': data or {},
        }
        self.save()

    def get_data(self, phase: str) -> dict:
        return self.state['phases'].get(phase, {}).get('data', {})

    def summary(self):
        n = len(self.state['phases'])
        print(f'Checkpoint: {self.exp_id} — {n} phase(s) completed')
        for phase, info in self.state['phases'].items():
            print(f'  ✓ {phase} ({info["timestamp"]})')

def start_periodic_sync(work_dir: Path, drive_output: Path, interval_s: int = 300):
    """Background thread that syncs checkpoint/manifest files to Drive every N seconds."""
    stop_event = threading.Event()
    sync_patterns = ['*.chk', '*.json', '*manifest*', '*.npz', '*.npy']
    def _sync():
        while not stop_event.is_set():
            stop_event.wait(interval_s)
            if stop_event.is_set():
                break
            for pat in sync_patterns:
                for f in work_dir.rglob(pat):
                    if f.is_file() and f.stat().st_size < 500_000_000:
                        dest = drive_output / f.relative_to(work_dir)
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        try:
                            shutil.copy2(f, dest)
                        except Exception:
                            pass
    t = threading.Thread(target=_sync, daemon=True, name='drive-sync')
    t.start()
    return stop_event

def restore_from_drive(drive_output: Path, work_dir: Path):
    """On session restart, copy checkpoint/manifest files from Drive back to local."""
    restored = 0
    for pat in ['*.chk', '*manifest*', '*.json']:
        for f in drive_output.rglob(pat):
            if f.is_file():
                dest = work_dir / f.relative_to(drive_output)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(f, dest)
                    restored += 1
    if restored:
        print(f'Restored {restored} checkpoint/manifest files from Drive')

# Initialize
ckpt = ExperimentCheckpoint(DRIVE_OUTPUT, EXP_ID)
restore_from_drive(DRIVE_OUTPUT, WORK_DIR)
sync_stop = start_periodic_sync(WORK_DIR, DRIVE_OUTPUT, interval_s=300)
ckpt.summary()
print('Auto-save: checkpoint/manifest files sync to Drive every 5 min')

In [ ]:
import openmm
from openmm import unit, Platform
Platform.getPlatformByName('CUDA')
print(f'OpenMM {openmm.__version__}, CUDA ready')

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mdtraj as md
from src.config import (SystemConfig, MinimizationConfig, EquilibrationConfig, FEPConfig, KCAL_TO_KJ)
from src.prep.pdb_fetch import fetch_pdb
from src.prep.pdb_clean import clean_structure
from src.prep.topology import build_topology
from src.prep.solvate import solvate_system
from src.simulate.minimizer import minimize_energy
from src.simulate.equilibrate import run_nvt, run_npt
from src.simulate.fep import run_fep_campaign
from src.simulate.platform import select_platform
from src.analyze.fep import compute_delta_g_mbar, compute_delta_delta_g
from openmm.app import PME, ForceField, Simulation, HBonds, PDBFile
from openmm import LangevinMiddleIntegrator, XmlSerializer
print('Imports OK')

In [ ]:
TEMPERATURE_K = 310.0
system_config = SystemConfig()
min_config = MinimizationConfig(max_iterations=10000)
eq_config = EquilibrationConfig(nvt_duration_ps=500.0, npt_duration_ps=1000.0, temperature_k=TEMPERATURE_K)
fep_config = FEPConfig(n_lambda_windows=20, per_window_duration_ns=2.0, temperature_k=TEMPERATURE_K)

In [ ]:
# Load SPINK1-Trypsin complex from EXP-06 or build fresh
PREP_DIR = WORK_DIR / 'prep'
EXP06_DIR = Path('/content/drive/MyDrive/v3_gpu_results/EXP-06')
exp06_sys = EXP06_DIR / 'system.xml'
exp06_state_candidates = [EXP06_DIR / 'eq_state.xml'] + list(EXP06_DIR.rglob('npt_final_state.xml'))
exp06_state = next((p for p in exp06_state_candidates if p.exists()), None)
topo_candidates = list(EXP06_DIR.rglob('topology_reference.pdb')) + list(EXP06_DIR.rglob('*solvated*.pdb'))

if exp06_sys.exists() and exp06_state:
    complex_system = XmlSerializer.deserialize(exp06_sys.read_text())
    complex_state_obj = XmlSerializer.deserialize(exp06_state.read_text())
    complex_positions = complex_state_obj.getPositions(asNumpy=True)
    complex_topo_pdb = str(topo_candidates[0]) if topo_candidates else None
    print('Complex loaded from EXP-06')
else:
    # Build SPINK1-Trypsin from 1TGS (SPINK1 is homologous to PSTI chain I)
    print('Building SPINK1-Trypsin from 1TGS...')
    raw_pdb = fetch_pdb('1TGS', output_dir=PREP_DIR)
    cleaned = clean_structure(raw_pdb, chains_to_keep=['E', 'I'], remove_heteroatoms=True, remove_waters=True)
    _, complex_system, modeller = build_topology(cleaned, system_config)
    modeller, nw, _, _ = solvate_system(modeller, system_config)
    ff = ForceField(system_config.force_field, system_config.water_model)
    complex_system = ff.createSystem(modeller.topology, nonbondedMethod=PME,
        nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds, rigidWater=True)
    intg = LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
    sim = Simulation(modeller.topology, complex_system, intg, select_platform('CUDA'))
    sim.context.setPositions(modeller.positions)
    complex_topo_pdb = str(PREP_DIR / 'spink1_trypsin.pdb')
    with open(complex_topo_pdb, 'w') as f: PDBFile.writeFile(modeller.topology, modeller.positions, f)
    minimize_energy(sim, min_config)
    run_nvt(sim, eq_config, WORK_DIR/'fep_complex'/'nvt')
    run_npt(sim, eq_config, WORK_DIR/'fep_complex'/'npt')
    complex_positions = sim.context.getState(getPositions=True).getPositions(asNumpy=True)
    print(f'Complex built and equilibrated ({nw} waters)')

In [ ]:
# Free leg: isolated SPINK1/PSTI in water
raw_inhib = fetch_pdb('1TGS', output_dir=PREP_DIR)
cleaned_inhib = clean_structure(raw_inhib, chains_to_keep=['I'], remove_heteroatoms=True, remove_waters=True)
_, free_system, mod_free = build_topology(cleaned_inhib, system_config)
mod_free, nw_f, _, _ = solvate_system(mod_free, system_config)
ff = ForceField(system_config.force_field, system_config.water_model)
free_system = ff.createSystem(mod_free.topology, nonbondedMethod=PME,
    nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds, rigidWater=True)
intg_f = LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
sim_f = Simulation(mod_free.topology, free_system, intg_f, select_platform('CUDA'))
sim_f.context.setPositions(mod_free.positions)
free_topo_pdb = str(PREP_DIR / 'spink1_free.pdb')
with open(free_topo_pdb, 'w') as f: PDBFile.writeFile(mod_free.topology, mod_free.positions, f)
minimize_energy(sim_f, min_config)
run_nvt(sim_f, eq_config, WORK_DIR/'fep_free'/'nvt')
run_npt(sim_f, eq_config, WORK_DIR/'fep_free'/'npt')
free_positions = sim_f.context.getState(getPositions=True).getPositions(asNumpy=True)
print(f'Free SPINK1 equilibrated')

In [ ]:
# Identify N34 sidechain atoms for Asn->Ser transformation
# For N->S: annihilate the sidechain atoms that differ (CG, OD1, ND2 + H's)
complex_top = md.load(complex_topo_pdb).topology
free_top = md.load(free_topo_pdb).topology

# Find SPINK1 chain in complex (the inhibitor chain)
complex_chains = list(complex_top.chains)
chain_sizes = [(c.index, len(list(c.atoms))) for c in complex_chains]
spink1_chain_cx = min(chain_sizes, key=lambda x: x[1])[0]

BACKBONE_NAMES = {'N', 'CA', 'C', 'O', 'H', 'HA', 'HA2', 'HA3', 'OXT', 'CB'}

def get_sidechain_beyond_cb(topology, resSeq, chain_id):
    indices = []
    for atom in topology.atoms:
        if atom.residue.resSeq == resSeq and atom.residue.chain.index == chain_id:
            if atom.name not in BACKBONE_NAMES:
                indices.append(atom.index)
    return indices

mutant_cx = get_sidechain_beyond_cb(complex_top, 34, spink1_chain_cx)
mutant_fr = get_sidechain_beyond_cb(free_top, 34, 0)

# Fallback: if no sidechain found, use all residue atoms
if not mutant_cx:
    mutant_cx = [a.index for a in complex_top.atoms if a.residue.resSeq == 34 and a.residue.chain.index == spink1_chain_cx]
if not mutant_fr:
    mutant_fr = [a.index for a in free_top.atoms if a.residue.resSeq == 34 and a.residue.chain.index == 0]

print(f'N34 complex atoms: {len(mutant_cx)}')
print(f'N34 free atoms: {len(mutant_fr)}')

In [ ]:
# Complex leg FEP
if ckpt.is_done('fep_complex'):
    print('⏭ Complex FEP already completed, skipping')
    dg_cx = ckpt.get_data('fep_complex')
else:
    fep_cx = run_fep_campaign(system=complex_system, positions=complex_positions,
        mutant_atom_indices=mutant_cx, config=fep_config,
        output_dir=WORK_DIR/'fep_complex'/'N34S')
    dg_cx = compute_delta_g_mbar(fep_cx['energy_matrix'], fep_cx['n_samples_per_state'], TEMPERATURE_K)
    print(f'Complex ΔG: {dg_cx["delta_g_kcal_mol"]:.2f} ± {dg_cx["delta_g_std_kcal_mol"]:.2f}')
    ckpt.mark_done('fep_complex', dg_cx)

In [ ]:
# Free leg FEP
if ckpt.is_done('fep_free'):
    print('⏭ Free FEP already completed, skipping')
    dg_fr = ckpt.get_data('fep_free')
else:
    fep_fr = run_fep_campaign(system=free_system, positions=free_positions,
        mutant_atom_indices=mutant_fr, config=fep_config,
        output_dir=WORK_DIR/'fep_free'/'N34S')
    dg_fr = compute_delta_g_mbar(fep_fr['energy_matrix'], fep_fr['n_samples_per_state'], TEMPERATURE_K)
    print(f'Free ΔG: {dg_fr["delta_g_kcal_mol"]:.2f} ± {dg_fr["delta_g_std_kcal_mol"]:.2f}')
    ckpt.mark_done('fep_free', dg_fr)

In [ ]:
ddg = compute_delta_delta_g(dg_cx, dg_fr)
ddg_val = ddg['delta_delta_g_kcal_mol']
ddg_std = ddg['delta_delta_g_std_kcal_mol']

print(f'\u0394\u0394G (N34S): {ddg_val:.2f} \u00b1 {ddg_std:.2f} kcal/mol')

abs_ddg = abs(ddg_val)
if abs_ddg < 1.0:
    verdict = 'PASS'
elif abs_ddg < 2.0:
    verdict = 'MARGINAL'
else:
    verdict = 'FAIL'
print(f'VERDICT: {verdict} (expected |\u0394\u0394G| < 1.0)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
profile_cx = dg_cx['free_energy_profile_kj_mol'] / KCAL_TO_KJ
profile_fr = dg_fr['free_energy_profile_kj_mol'] / KCAL_TO_KJ
lam = np.linspace(0, 1, len(profile_cx))
ax.plot(lam, profile_cx, 'b-o', ms=4, label='Complex')
ax.plot(lam, profile_fr, 'r-s', ms=4, label='Free')
ax.set_xlabel('\u03bb'); ax.set_ylabel('\u0394G (kcal/mol)'); ax.legend()
ax.set_title('Alchemical Profile')

ax = axes[1]
ax.bar(['\u0394\u0394G'], [ddg_val], yerr=[ddg_std], color='steelblue', edgecolor='navy', capsize=10)
ax.axhspan(-1, 1, alpha=0.15, color='green', label='PASS: |\u0394\u0394G| < 1.0')
ax.axhline(0, color='k', ls='-', lw=0.5)
ax.set_ylabel('kcal/mol'); ax.set_title(f'N34S \u0394\u0394G = {ddg_val:.2f} \u2014 {verdict}'); ax.legend()

fig.suptitle('EXP-32: SPINK1 N34S Negative Control', fontsize=14)
fig.tight_layout(); fig.savefig(WORK_DIR/'figures'/'results.png', dpi=150); plt.close(fig)

In [ ]:
sync_stop.set()  # Stop periodic sync before final copy

results = {
    'experiment_id': EXP_ID, 'ddg_kcal': round(ddg_val, 3), 'ddg_std_kcal': round(ddg_std, 3),
    'abs_ddg': round(abs_ddg, 3),
    'dg_complex_kcal': round(dg_cx['delta_g_kcal_mol'], 3),
    'dg_free_kcal': round(dg_fr['delta_g_kcal_mol'], 3), 'verdict': verdict,
}
with open(WORK_DIR/'results.json', 'w') as f: json.dump(results, f, indent=2)

for item in WORK_DIR.rglob('*'):
    if item.is_file():
        dest = DRIVE_OUTPUT / item.relative_to(WORK_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)

ckpt.mark_done('complete')

# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file(): zf.write(item, f'{EXP_ID}/{item.relative_to(WORK_DIR)}')
files.download(str(zip_path))